# Qwen2.5 1.5B QLoRA for ECtHR Alleged-Violation Prediction

This notebook fine-tunes a small local model for the same target used in the EU traversal notebook: `ECTHR_CONFIG = "alleged-violation-prediction"`.

The default mode trains a retrieval-grounded selector: it receives case facts plus candidate Convention article chunks, then outputs the alleged article IDs in JSON. This matches the final selector step in `eu_conventions_tree_traversal_examples.ipynb`.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import gc
import json
import random
import re
import sys

import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'local_retrieval_model.py').exists():
    for candidate in [
        NOTEBOOK_DIR / 'llm-guided-retrieval' / 'src' / 'llm_rl_playground',
        NOTEBOOK_DIR / 'src' / 'llm_rl_playground',
        NOTEBOOK_DIR.parent / 'llm_rl_playground',
    ]:
        if (candidate / 'local_retrieval_model.py').exists():
            NOTEBOOK_DIR = candidate
            break

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f'Using notebook directory: {NOTEBOOK_DIR}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Config

`TRAINING_MODE = "selector"` trains on facts plus candidate articles. Use `"direct"` if you want facts-only prediction.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

ECTHR_DATASET = 'AUEB-NLP/ecthr_cases'
ECTHR_CONFIG = 'alleged-violation-prediction'
ECTHR_TRAIN_SPLIT = 'train'
ECTHR_EVAL_SPLIT = 'validation'

CHUNKS_PATH = NOTEBOOK_DIR.parent / 'tree_construction' / 'EU_conventions_example' / 'Convention_ENG_chunks.json'
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'qwen2.5-1.5b-ecthr-alleged-qlora'

TRAINING_MODE = 'selector'  # 'selector' or 'direct'
SEED = 42
MAX_TRAIN_EXAMPLES = 2000
MAX_EVAL_EXAMPLES = 200
MAX_FACT_CHARS = 9000
MAX_EVIDENCE_CHARS = 900
NUM_NEGATIVE_ARTICLES = 8
MAX_LENGTH = 2048

NUM_TRAIN_EPOCHS = 1.0
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Set this to OUTPUT_DIR / 'checkpoint-*' to resume the same run with optimizer state.
RESUME_FROM_CHECKPOINT = None

assert TRAINING_MODE in {'selector', 'direct'}
if TRAINING_MODE == 'selector':
    assert CHUNKS_PATH.exists(), f'Missing Convention chunks: {CHUNKS_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    'model': MODEL_ID,
    'ecthr_config': ECTHR_CONFIG,
    'training_mode': TRAINING_MODE,
    'output': str(OUTPUT_DIR),
})

## 3. Label and article helpers

In [ ]:
def clean_space(value):
    return ' '.join(str(value or '').split())


def normalize_article_label(value):
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None

    s = s.replace('no violation', '')
    s = s.replace('non violation', '')
    s = s.replace('non-violation', '')
    s = s.replace('.', '')
    s = s.replace('-', '_')
    s = re.sub(r'\s+', '_', s)
    s = re.sub(r'^echr_', '', s)
    s = re.sub(r'^convention_', '', s)

    if re.fullmatch(r'\d+[a-z]?', s):
        return f'article_{s}'

    m = re.fullmatch(r'p(\d+)_(\d+)', s)
    if m:
        return f'protocol_{int(m.group(1))}_article_{int(m.group(2))}'

    m = re.search(r'article_?(\d+[a-z]?)_(?:of_)?protocol_?(\d+)', s)
    if m:
        return f'protocol_{int(m.group(2))}_article_{m.group(1)}'

    m = re.search(r'protocol_?(\d+)_article_?(\d+[a-z]?)', s)
    if m:
        return f'protocol_{int(m.group(1))}_article_{m.group(2)}'

    m = re.search(r'article_?(\d+[a-z]?)', s)
    if m:
        return f'article_{m.group(1)}'

    return None


def article_id_to_display(article_id):
    m = re.fullmatch(r'article_(\d+[a-z]?)', article_id)
    if m:
        return f'Article {m.group(1).upper()}'
    m = re.fullmatch(r'protocol_(\d+)_article_(\d+[a-z]?)', article_id)
    if m:
        return f'Protocol {m.group(1)}, Article {m.group(2).upper()}'
    return article_id


TREE_ARTICLE_RE = re.compile(
    r'(?:Protocol\s+(?P<protocol>\d+)\s+)?Art\.\s*(?P<article>\d+[A-Za-z]?)',
    flags=re.IGNORECASE,
)


def article_id_from_chunk(chunk):
    text = ' '.join(
        clean_space(chunk.get(key))
        for key in ('citation', 'article_number', 'article_title', 'text')
    )
    m = TREE_ARTICLE_RE.search(text)
    if not m:
        return normalize_article_label(chunk.get('citation') or chunk.get('article_number'))
    article = m.group('article').lower()
    protocol = m.group('protocol')
    if protocol:
        return f'protocol_{int(protocol)}_article_{article}'
    return f'article_{article}'


def example_gold_articles(example, label_column='labels'):
    raw_labels = example.get(label_column, [])
    if raw_labels is None:
        return []
    if isinstance(raw_labels, (int, str)):
        raw_labels = [raw_labels]
    normalized = []
    for raw_label in raw_labels:
        article_id = normalize_article_label(raw_label)
        if article_id and article_id not in normalized:
            normalized.append(article_id)
    return sorted(normalized)


def facts_to_text(example, max_chars=MAX_FACT_CHARS):
    facts = example.get('text') or example.get('facts') or []
    if isinstance(facts, str):
        fact_text = facts
    else:
        fact_text = '\n'.join(f'- {clean_space(fact)}' for fact in facts if clean_space(fact))
    if len(fact_text) > max_chars:
        fact_text = fact_text[:max_chars] + '\n- [facts truncated]'
    return fact_text

## 4. Load ECtHR cases and Convention chunks

In [ ]:
train_cases = load_dataset(
    ECTHR_DATASET,
    ECTHR_CONFIG,
    split=ECTHR_TRAIN_SPLIT,
    trust_remote_code=True,
)

try:
    eval_cases = load_dataset(
        ECTHR_DATASET,
        ECTHR_CONFIG,
        split=ECTHR_EVAL_SPLIT,
        trust_remote_code=True,
    )
except Exception:
    eval_cases = None

print(train_cases)
print('First labels:', train_cases[0]['labels'])
print('First normalized labels:', example_gold_articles(train_cases[0]))

In [ ]:
def load_article_chunk_pool(path):
    with path.open('r', encoding='utf-8') as f:
        payload = json.load(f)

    pool = {}
    for chunk in payload['chunks']:
        text = clean_space(chunk.get('text'))
        if len(text) < 40:
            continue
        article_id = article_id_from_chunk(chunk)
        if not article_id:
            continue
        pool.setdefault(article_id, []).append({
            'article_id': article_id,
            'label': article_id_to_display(article_id),
            'citation': clean_space(chunk.get('citation')),
            'text': text,
        })
    return pool


article_pool = load_article_chunk_pool(CHUNKS_PATH) if TRAINING_MODE == 'selector' else {}
print(f'Article IDs with chunks: {len(article_pool)}')
print(sorted(article_pool)[:20])

## 5. Convert cases into supervised chat examples

In [ ]:
def make_direct_example(example):
    gold = example_gold_articles(example)
    if not gold:
        return None

    user = f"""Given the facts of an ECtHR case, predict the Convention or Protocol articles alleged by the applicant.

Return only JSON with this shape:
{{"reasoning": "brief explanation", "selected_articles": ["article_..."]}}

Case facts:
{facts_to_text(example)}"""

    assistant = json.dumps({
        'reasoning': 'The selected articles are the alleged Convention or Protocol provisions associated with the case facts.',
        'selected_articles': gold,
    }, ensure_ascii=False)

    return {
        'messages': [
            {'role': 'system', 'content': 'You predict alleged ECtHR article IDs from case facts. Return only valid JSON.'},
            {'role': 'user', 'content': user},
            {'role': 'assistant', 'content': assistant},
        ]
    }


def make_selector_example(example, rng):
    gold_all = example_gold_articles(example)
    gold_with_chunks = [article_id for article_id in gold_all if article_id in article_pool]
    if not gold_with_chunks:
        return None

    candidate_ids = list(gold_with_chunks)
    negative_ids = [article_id for article_id in article_pool if article_id not in set(gold_all)]
    rng.shuffle(negative_ids)
    candidate_ids.extend(negative_ids[:NUM_NEGATIVE_ARTICLES])
    rng.shuffle(candidate_ids)

    candidate_lines = []
    for article_id in candidate_ids:
        chunk = rng.choice(article_pool[article_id])
        candidate_lines.append(
            f'- {article_id} ({chunk["label"]})\n'
            f'  Evidence: {chunk["text"][:MAX_EVIDENCE_CHARS]}'
        )

    user = f"""You are doing the final ECtHR alleged-violation article selection after a LATTICE retrieval step.

Choose only candidate article IDs that are likely to be alleged violations for this case. Do not choose articles that are merely background, procedural context, or weakly related.

Return JSON with:
- reasoning: a brief explanation
- selected_articles: normalized IDs copied exactly from the candidate list

Case facts:
{facts_to_text(example)}

Candidate articles from LATTICE:
{chr(10).join(candidate_lines)}"""

    selected = sorted(article_id for article_id in candidate_ids if article_id in set(gold_all))
    assistant = json.dumps({
        'reasoning': 'The selected candidate article IDs correspond to the alleged Convention or Protocol provisions in the case labels.',
        'selected_articles': selected,
    }, ensure_ascii=False)

    return {
        'messages': [
            {'role': 'system', 'content': 'You are a local legal retrieval selector. Return only valid JSON.'},
            {'role': 'user', 'content': user},
            {'role': 'assistant', 'content': assistant},
        ]
    }


def build_rows(dataset, max_examples, seed):
    rng = random.Random(seed)
    indexes = list(range(len(dataset)))
    rng.shuffle(indexes)

    rows = []
    for idx in indexes:
        example = dataset[idx]
        row = make_selector_example(example, rng) if TRAINING_MODE == 'selector' else make_direct_example(example)
        if row is not None:
            rows.append(row)
        if len(rows) >= max_examples:
            break
    return rows


train_rows = build_rows(train_cases, MAX_TRAIN_EXAMPLES, SEED)
eval_rows = build_rows(eval_cases, MAX_EVAL_EXAMPLES, SEED + 1) if eval_cases is not None else []

raw_train_dataset = Dataset.from_list(train_rows)
raw_eval_dataset = Dataset.from_list(eval_rows) if eval_rows else None

print({'train_examples': len(raw_train_dataset), 'eval_examples': 0 if raw_eval_dataset is None else len(raw_eval_dataset)})
print(raw_train_dataset[0]['messages'][1]['content'][:1500])
print(raw_train_dataset[0]['messages'][2]['content'])

## 6. Tokenize with loss only on the assistant answer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


def preprocess(example):
    messages = example['messages']
    prompt_messages = messages[:-1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = full_tokens['input_ids']
    labels = input_ids.copy()
    prompt_len = min(len(prompt_tokens['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    return {
        'input_ids': input_ids,
        'attention_mask': full_tokens['attention_mask'],
        'labels': labels,
    }


train_dataset = raw_train_dataset.map(
    preprocess,
    remove_columns=raw_train_dataset.column_names,
    desc='Tokenizing train examples',
)
train_dataset = train_dataset.filter(
    lambda example: any(label != -100 for label in example['labels']),
    desc='Dropping fully masked train examples',
)

eval_dataset = None
if raw_eval_dataset is not None:
    eval_dataset = raw_eval_dataset.map(
        preprocess,
        remove_columns=raw_eval_dataset.column_names,
        desc='Tokenizing eval examples',
    )
    eval_dataset = eval_dataset.filter(
        lambda example: any(label != -100 for label in example['labels']),
        desc='Dropping fully masked eval examples',
    )

print({'train': len(train_dataset), 'eval': 0 if eval_dataset is None else len(eval_dataset)})

## 7. Load Qwen2.5 in 4-bit and attach LoRA

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('QLoRA training expects a CUDA GPU.')

torch.manual_seed(SEED)
compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

gc.collect()
torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    device_map={'': 0},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules='all-linear',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 8. Train and save the adapter

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    eval_strategy='epoch' if eval_dataset is not None else 'no',
    optim='paged_adamw_8bit',
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    warmup_ratio=0.03,
    lr_scheduler_type='constant',
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
    ),
)

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print(f'Saved QLoRA adapter to: {OUTPUT_DIR}')

## 9. Smoke test

In [ ]:
def generate_reply(model, tokenizer, messages, max_new_tokens=256):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    )
    prompt = {key: value.to(model.device) for key, value in prompt.items()}

    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(
            **prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = prompt['input_ids'].shape[-1]
    new_tokens = generated_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


test_messages = raw_train_dataset[0]['messages'][:-1]
print(generate_reply(model, tokenizer, test_messages))
print('Gold:', raw_train_dataset[0]['messages'][-1]['content'])

## 10. Load with `LocalRetrievalModelRuntime`

In [ ]:
from local_retrieval_model import load_local_retrieval_model

del trainer
del model
gc.collect()
torch.cuda.empty_cache()

runtime = load_local_retrieval_model(
    model_id=MODEL_ID,
    adapter_path=OUTPUT_DIR,
    use_4bit=True,
)

sample_user_prompt = raw_train_dataset[0]['messages'][1]['content']
reply = runtime.ask(
    sample_user_prompt,
    system_prompt=raw_train_dataset[0]['messages'][0]['content'],
    max_new_tokens=256,
    temperature=0.0,
)
print(reply)